# Hệ thống hội thoại hướng tác vụ (Task-Oriented Dialogue) end-to-end
## Bài toán: Hỏi đáp – gợi ý – tìm kiếm sách (Tiếng Việt)
**Môn HCI** · Mô hình: Transformer seq2seq tiền huấn luyện (ViT5 / BARTpho / mT5) + tùy chọn LLM (Qwen / Llama + LoRA)

---
### 1. Ý tưởng tổng quan
Hệ thống TOD cổ điển là **pipeline**: NLU → DST (theo dõi trạng thái) → Policy → NLG. Mỗi module huấn luyện riêng nên lỗi dồn tích.
Cách tiếp cận **end-to-end** (SimpleTOD, UBAR, MinTL, ...) dùng **một mô hình ngôn ngữ duy nhất** sinh ra toàn bộ chuỗi:

```
Lịch sử hội thoại + câu người dùng
          │
   [Transformer seq2seq]  ──►  (1) Belief state:  genre: trinh thám ; rating_min: 4.5
          │                                  │
          │                          [Truy vấn CSDL sách]  ──►  n kết quả
          │                                  │
   [Transformer seq2seq] ◄───────────────────┘
          │
          └─► (2) Dialogue act + câu trả lời đã "delexicalize"
                    │
              điền thực thể thật từ CSDL ──►  câu trả lời cuối cùng
```

* **Một mô hình, hai tác vụ** (phân biệt bằng prefix `dst:` và `resp:`), huấn luyện chung → không cần module NLU/Policy/NLG riêng.
* **Delexicalization**: câu trả lời dùng placeholder như `[title_1]`, `[price_1]`; thực thể thật lấy từ CSDL → mô hình không "bịa" tên sách/giá.
* **Knowledge base (KB)**: bảng sách có `title, author, genre, year, rating, price`.

### 2. Nội dung notebook
0. Cài đặt & cấu hình
1. Xây dựng CSDL sách + hàm truy vấn
2. Sinh bộ dữ liệu hội thoại nhiều lượt (có nhãn belief state / act)
3. Chuyển dữ liệu sang dạng seq2seq
4. Fine-tune mô hình (T5 / BART)
5. Suy luận & chatbot hội thoại
6. Đánh giá (JGA, Slot-F1, Act-Acc, BLEU, Inform, Success)
7. (Tùy chọn) Fine-tune LLM decoder-only bằng LoRA
8. Đánh giá người dùng cho môn HCI (SUS)
9. Hướng mở rộng

> **Lưu ý:** Chạy trên GPU (Colab T4 là đủ cho `vit5-base`; máy yếu dùng `google/mt5-small`). Dữ liệu hội thoại, `rating` và `price` trong notebook là **mô phỏng** để huấn luyện/demo; khi viết báo cáo hãy nói rõ điều này và nên thay bằng dữ liệu thật (xem mục 9).

## 0. Cài đặt & cấu hình

In [1]:
# Bỏ comment dòng dưới khi chạy trên Colab / môi trường mới
!pip install -q sacrebleu sentencepiece protobuf
# Kaggle: Settings > Internet = On (bắt buộc để tải mô hình), Accelerator = GPU.
# Không cần cài/nâng cấp transformers, torch (Kaggle đã có sẵn). 

In [2]:
import os, re, json, math, random, difflib, collections
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ============ CẤU HÌNH ============
MODEL_NAME  = "VietAI/vit5-base"   # gợi ý khác: "vinai/bartpho-syllable", "google/mt5-small"
MAX_SRC     = 384                  # độ dài tối đa đầu vào (token)
MAX_TGT     = 96                   # độ dài tối đa đầu ra (token)
EPOCHS      = 5
BATCH       = 16
N_DIALOGUES = 2000                 # số hội thoại mô phỏng
LR          = 3e-4 if "t5" in MODEL_NAME.lower() else 5e-5   # T5 cần lr lớn hơn BART
OUT_DIR     = "/kaggle/working/tod-books" if os.path.exists("/kaggle/working") else "outputs/tod-books"

## 1. Cơ sở dữ liệu sách (Knowledge Base)
Tên sách / tác giả / thể loại / năm xuất bản bên dưới là dữ liệu mẫu (năm có thể là năm xuất bản lần đầu theo nguyên tác, chỉ mang tính minh họa). `rating` và `price` (nghìn đồng) được **sinh ngẫu nhiên có seed cố định**.

Hàm `BookDB.query(belief_state)` là "công cụ" mà hệ thống gọi giữa hai bước sinh: lọc theo thể loại, tác giả, đánh giá tối thiểu, giá tối đa, năm xuất bản tối thiểu, hoặc tìm theo tên sách (có fuzzy matching); `pick` chọn cuốn thứ *i* trong danh sách đã gợi ý.

In [3]:
RAW_BOOKS = [
    # (title, author, genre, year)
    ("Mắt Biếc", "Nguyễn Nhật Ánh", "văn học", 1990),
    ("Tôi Thấy Hoa Vàng Trên Cỏ Xanh", "Nguyễn Nhật Ánh", "văn học", 2010),
    ("Cho Tôi Xin Một Vé Đi Tuổi Thơ", "Nguyễn Nhật Ánh", "thiếu nhi", 2008),
    ("Cô Gái Đến Từ Hôm Qua", "Nguyễn Nhật Ánh", "tình cảm", 1989),
    ("Dế Mèn Phiêu Lưu Ký", "Tô Hoài", "thiếu nhi", 1941),
    ("Số Đỏ", "Vũ Trọng Phụng", "văn học", 1936),
    ("Cánh Đồng Bất Tận", "Nguyễn Ngọc Tư", "văn học", 2005),
    ("Nhà Giả Kim", "Paulo Coelho", "văn học", 1988),
    ("Bố Già", "Mario Puzo", "văn học", 1969),
    ("Rừng Na Uy", "Haruki Murakami", "tình cảm", 1987),
    ("Hai Số Phận", "Jeffrey Archer", "tình cảm", 1979),
    ("Sherlock Holmes Toàn Tập", "Arthur Conan Doyle", "trinh thám", 1892),
    ("Án Mạng Trên Chuyến Tàu Tốc Hành Phương Đông", "Agatha Christie", "trinh thám", 1934),
    ("Và Rồi Chẳng Còn Ai", "Agatha Christie", "trinh thám", 1939),
    ("Mật Mã Da Vinci", "Dan Brown", "trinh thám", 2003),
    ("Nghi Can X", "Higashino Keigo", "trinh thám", 2005),
    ("Bạch Dạ Hành", "Higashino Keigo", "trinh thám", 1999),
    ("Dune", "Frank Herbert", "khoa học viễn tưởng", 1965),
    ("1984", "George Orwell", "khoa học viễn tưởng", 1949),
    ("Hai Vạn Dặm Dưới Đáy Biển", "Jules Verne", "khoa học viễn tưởng", 1870),
    ("Ender Trò Chơi Chiến Tranh", "Orson Scott Card", "khoa học viễn tưởng", 1985),
    ("Tam Thể", "Lưu Từ Hân", "khoa học viễn tưởng", 2008),
    ("Cha Giàu Cha Nghèo", "Robert Kiyosaki", "kinh tế", 1997),
    ("Tư Duy Nhanh Và Chậm", "Daniel Kahneman", "kinh tế", 2011),
    ("Nghĩ Giàu Làm Giàu", "Napoleon Hill", "kinh tế", 1937),
    ("Người Giàu Có Nhất Thành Babylon", "George Clason", "kinh tế", 1926),
    ("Khởi Nghiệp Tinh Gọn", "Eric Ries", "kinh tế", 2011),
    ("Đắc Nhân Tâm", "Dale Carnegie", "tâm lý", 1936),
    ("Nghệ Thuật Tinh Tế Của Việc Đếch Quan Tâm", "Mark Manson", "tâm lý", 2016),
    ("Sức Mạnh Của Thói Quen", "Charles Duhigg", "tâm lý", 2012),
    ("Dám Bị Ghét", "Kishimi Ichiro", "tâm lý", 2013),
    ("Phi Lý Trí", "Dan Ariely", "tâm lý", 2008),
    ("Hiểu Về Trái Tim", "Minh Niệm", "tâm lý", 2011),
    ("Sapiens Lược Sử Loài Người", "Yuval Noah Harari", "lịch sử", 2011),
    ("Súng Vi Trùng Và Thép", "Jared Diamond", "lịch sử", 1997),
    ("Việt Nam Sử Lược", "Trần Trọng Kim", "lịch sử", 1920),
    ("Lược Sử Thời Gian", "Stephen Hawking", "khoa học", 1988),
    ("Vũ Trụ Trong Vỏ Hạt Dẻ", "Stephen Hawking", "khoa học", 2001),
    ("Cosmos", "Carl Sagan", "khoa học", 1980),
    ("Hoàng Tử Bé", "Antoine de Saint-Exupéry", "thiếu nhi", 1943),
    ("Harry Potter Và Hòn Đá Phù Thủy", "J.K. Rowling", "thiếu nhi", 1997),
    ("Totto-chan Bên Cửa Sổ", "Tetsuko Kuroyanagi", "thiếu nhi", 1981),
    ("Clean Code", "Robert C. Martin", "công nghệ", 2008),
    ("The Pragmatic Programmer", "Andrew Hunt", "công nghệ", 1999),
    ("Introduction To Algorithms", "Thomas Cormen", "công nghệ", 1990),
    ("Deep Learning", "Ian Goodfellow", "công nghệ", 2016),
    ("Designing Data-Intensive Applications", "Martin Kleppmann", "công nghệ", 2017),
    ("Don't Make Me Think", "Steve Krug", "công nghệ", 2000),
    ("The Design Of Everyday Things", "Don Norman", "công nghệ", 1988),
]

_rng = random.Random(0)
BOOKS = []
for i, (t, a, g, y) in enumerate(RAW_BOOKS):
    BOOKS.append(dict(id=i, title=t, author=a, genre=g, year=y,
                      rating=round(_rng.uniform(3.8, 4.8), 1),     # MÔ PHỎNG
                      price=_rng.randrange(60, 260, 5)))           # MÔ PHỎNG (nghìn đồng)
print(len(BOOKS), "cuốn sách |", len({b['genre'] for b in BOOKS}), "thể loại")
print(BOOKS[0])

49 cuốn sách | 10 thể loại
{'id': 0, 'title': 'Mắt Biếc', 'author': 'Nguyễn Nhật Ánh', 'genre': 'văn học', 'year': 1990, 'rating': 4.6, 'price': 190}


In [4]:
def norm(s):
    return re.sub(r"\s+", " ", s.lower().strip())

FILTER_SLOTS = ["genre", "author", "rating_min", "price_max", "year_min"]
SLOT_ORDER   = ["title", "genre", "author", "rating_min", "price_max", "year_min", "pick", "ask"]
# title/genre/author/rating_min/price_max/year_min: điều kiện tìm kiếm (lưu lại qua các lượt)
# pick: người dùng chọn cuốn thứ mấy trong danh sách vừa gợi ý (chỉ có hiệu lực 1 lượt)
# ask : thông tin người dùng đang hỏi (price/author/rating/year/genre)  (chỉ có hiệu lực 1 lượt)


class BookDB:
    def __init__(self, books):
        self.books = books
        self.titles = [norm(b["title"]) for b in books]

    def query(self, bs):
        """bs: dict belief state -> danh sách sách (đã sắp xếp theo rating giảm dần)."""
        if not any(k in bs for k in FILTER_SLOTS + ["title"]):
            return []
        rows = self.books
        if "title" in bs:
            t = norm(bs["title"])
            rows = [b for b in rows if t in norm(b["title"])]
            if not rows:                                   # fuzzy match khi gõ sai nhẹ
                m = difflib.get_close_matches(t, self.titles, n=1, cutoff=0.8)
                rows = [b for b in self.books if norm(b["title"]) == m[0]] if m else []
        try:
            if "genre" in bs:       rows = [b for b in rows if b["genre"] == norm(bs["genre"])]
            if "author" in bs:      rows = [b for b in rows if norm(bs["author"]) in norm(b["author"])]
            if "rating_min" in bs:  rows = [b for b in rows if b["rating"] >= float(bs["rating_min"])]
            if "price_max" in bs:   rows = [b for b in rows if b["price"] <= float(bs["price_max"])]
            if "year_min" in bs:    rows = [b for b in rows if b["year"] >= int(float(bs["year_min"]))]
        except ValueError:
            rows = []
        rows = sorted(rows, key=lambda b: (-b["rating"], b["id"]))
        if "pick" in bs:
            try:
                i = int(bs["pick"])
                if 1 <= i <= len(rows):
                    rows = [rows[i - 1]]
            except ValueError:
                pass
        return rows


db = BookDB(BOOKS)
print([b["title"] for b in db.query({"genre": "trinh thám", "rating_min": "4.0"})])


def lexicalize(text, rows):
    """Điền thực thể thật từ kết quả DB vào các placeholder [title_1], [price_2], [count] ..."""
    text = text.replace("[count]", str(len(rows)))
    def repl(m):
        f, i = m.group(1), int(m.group(2))
        if i <= len(rows):
            v = rows[i - 1][f]
            return f"{v:.1f}" if f == "rating" else str(v)
        return m.group(0)
    return re.sub(r"\[(title|author|genre|year|rating|price)_(\d)\]", repl, text)

['Bạch Dạ Hành', 'Án Mạng Trên Chuyến Tàu Tốc Hành Phương Đông', 'Nghi Can X', 'Và Rồi Chẳng Còn Ai', 'Mật Mã Da Vinci']


## 2. Sinh dữ liệu hội thoại nhiều lượt
Chưa có sẵn bộ dữ liệu TOD tiếng Việt cho miền sách, nên ta **mô phỏng** bằng *user simulator + system policy* dựa trên luật và template (có biến thể câu để mô hình không chỉ học thuộc lòng). Mỗi lượt gồm:

| Trường | Ý nghĩa |
|---|---|
| `user` | câu người dùng |
| `bs` | belief state **tích lũy** (vd. `{genre: trinh thám, rating_min: 4.5}`) |
| `n` | số kết quả trả về từ DB khi truy vấn bằng `bs` |
| `act` | hành động của hệ thống: `request_genre`, `recommend`, `no_match`, `inform_book`, `no_match_title`, `inform_detail`, `bye` |
| `resp` | câu trả lời **delexicalized** (mục tiêu huấn luyện) |
| `resp_lex` | câu trả lời đã điền thực thể (đưa vào lịch sử hội thoại) |

Các hành vi người dùng được mô phỏng: chào hỏi, tìm theo thể loại/tác giả/đánh giá/giá/năm, **lọc thêm** điều kiện, hỏi chi tiết "cuốn thứ hai giá bao nhiêu?", tìm theo **tên sách**, đổi chủ đề, truy vấn **không có kết quả**, và kết thúc.

In [5]:
GREETS   = ["Xin chào", "Chào bạn", "Hi", "Chào shop", "Hello"]
BYES     = ["Cảm ơn bạn, mình đủ thông tin rồi", "Cảm ơn nhé, tạm biệt", "Ok cảm ơn, bye bạn", "Vậy là đủ rồi, cảm ơn nhiều"]
OPEN_ASK = ["Mình muốn tìm", "Gợi ý giúp mình", "Bạn có thể gợi ý", "Cho mình xin gợi ý", "Tìm giúp mình", "Mình đang cần tìm"]
REFINE_PRE = ["Lọc thêm giúp mình", "Mình muốn", "Cho mình xem loại", "Thu hẹp lại", "Ok, thêm điều kiện"]
TITLE_Q  = ["Bạn có cuốn {t} không?", "Tìm giúp mình cuốn {t}", "Cho mình xem thông tin cuốn {t}", "Cuốn {t} thế nào?"]
FAKE_TITLES = ["Bí Mật Của Gió Đêm", "Hành Trình Về Phía Mặt Trời", "Thành Phố Không Ngủ Yên", "Con Đường Tơ Lụa Số"]

ASK_PHRASES = {
    "price":  ["giá bao nhiêu", "bao nhiêu tiền", "giá thế nào"],
    "author": ["ai viết", "của tác giả nào", "tác giả là ai"],
    "rating": ["đánh giá thế nào", "được mấy sao", "có hay không"],
    "year":   ["xuất bản năm nào", "ra mắt khi nào"],
    "genre":  ["thuộc thể loại gì", "là thể loại nào"],
}
ORDINALS = {1: ["cuốn đầu tiên", "quyển thứ nhất", "cuốn số 1"],
            2: ["cuốn thứ hai", "quyển số 2", "cuốn thứ 2"],
            3: ["cuốn thứ ba", "quyển số 3", "cuốn thứ 3"]}

# ---- Template câu trả lời (delexicalized) ----
T_GREET = ["Xin chào! Mình là trợ lý tìm sách. Bạn đang quan tâm thể loại nào ạ?",
           "Chào bạn! Bạn muốn mình gợi ý sách thể loại nào (văn học, trinh thám, kinh tế, tâm lý, ...)?"]
T_BYE = ["Cảm ơn bạn! Chúc bạn đọc sách vui vẻ.", "Rất vui được giúp bạn. Hẹn gặp lại nhé!"]
T_NOMATCH = ["Rất tiếc, mình không tìm thấy cuốn nào phù hợp với yêu cầu. Bạn thử nới lỏng điều kiện như giá hoặc đánh giá nhé?",
             "Hiện chưa có cuốn nào đáp ứng đủ điều kiện bạn đưa ra. Bạn muốn đổi thể loại hoặc điều kiện khác không?"]
T_NOTITLE = ["Rất tiếc, mình không tìm thấy cuốn sách đó trong kho. Bạn thử kiểm tra lại tên sách hoặc để mình gợi ý theo thể loại nhé?"]
T_BOOK = ["Mình tìm thấy [title_1] của [author_1], thuộc thể loại [genre_1], xuất bản năm [year_1]. Sách được đánh giá [rating_1]/5 sao, giá [price_1] nghìn đồng.",
          "Có ạ! [title_1] của [author_1] ([genre_1], [year_1]), đánh giá [rating_1]/5, giá [price_1] nghìn đồng."]
T_REC1 = ["Mình tìm thấy 1 cuốn phù hợp: [title_1] của [author_1], đánh giá [rating_1]/5 sao. Bạn có muốn xem chi tiết không?",
          "Chỉ có 1 cuốn đáp ứng yêu cầu là [title_1] của [author_1], giá [price_1] nghìn đồng. Bạn thấy sao?"]
T_REC2 = ["Mình tìm thấy [count] cuốn phù hợp: [title_1] của [author_1] và [title_2] của [author_2]. Bạn muốn xem chi tiết cuốn nào?",
          "Có [count] cuốn đáp ứng yêu cầu: [title_1] ([author_1]) và [title_2] ([author_2]). Bạn cần mình lọc thêm gì không?"]
T_REC3 = ["Mình tìm thấy [count] cuốn phù hợp. Top gợi ý: [title_1] của [author_1], [title_2] của [author_2] và [title_3] của [author_3]. Bạn muốn xem chi tiết cuốn nào hay lọc thêm theo giá, đánh giá?",
          "Có [count] cuốn đáp ứng yêu cầu của bạn, nổi bật là [title_1] ([author_1]), [title_2] ([author_2]) và [title_3] ([author_3]). Bạn cần mình lọc thêm điều kiện nào không?"]
T_DETAIL = {
    "price":  ["Cuốn [title_1] có giá [price_1] nghìn đồng.", "[title_1] hiện có giá [price_1] nghìn đồng ạ."],
    "author": ["[title_1] là sáng tác của [author_1].", "Tác giả của [title_1] là [author_1] ạ."],
    "rating": ["[title_1] được đánh giá [rating_1]/5 sao.", "Điểm đánh giá của [title_1] là [rating_1]/5 sao ạ."],
    "year":   ["[title_1] xuất bản năm [year_1].", "Cuốn [title_1] ra mắt năm [year_1] ạ."],
    "genre":  ["[title_1] thuộc thể loại [genre_1].", "Cuốn [title_1] thuộc thể loại [genre_1] ạ."],
}


class DialogueGenerator:
    def __init__(self, db, seed=0):
        self.db, self.r = db, random.Random(seed)

    # ---------- cụm từ mô tả slot ----------
    def phrase(self, slot, val):
        r = self.r
        if slot == "genre":      return r.choice([f"thể loại {val}", f"về chủ đề {val}", f"{val}"])
        if slot == "author":     return r.choice([f"của tác giả {val}", f"của {val}", f"do {val} viết"])
        if slot == "rating_min": return r.choice([f"đánh giá từ {val} sao trở lên", f"rating trên {val}", f"điểm đánh giá ít nhất {val}"])
        if slot == "price_max":  return r.choice([f"giá dưới {val} nghìn", f"không quá {val}k", f"giá tối đa {val}k"])
        if slot == "year_min":   return r.choice([f"xuất bản sau năm {val}", f"ra mắt từ năm {val}"])

    def value_from_book(self, slot, b):
        """Chọn giá trị slot sao cho cuốn b chắc chắn thỏa điều kiện."""
        r = self.r
        if slot == "genre":      return b["genre"]
        if slot == "author":     return b["author"]
        if slot == "rating_min": return f"{r.choice([x for x in (3.5, 4.0, 4.2, 4.5) if x <= b['rating']]):.1f}"
        if slot == "price_max":  return str(int(math.ceil(b["price"] / 10) * 10 + r.choice([0, 10, 30])))
        if slot == "year_min":   return str(b["year"] // 10 * 10)

    # ---------- câu người dùng ----------
    def opening(self, book, allow_nomatch=True):
        r = self.r
        slots = {}
        if r.random() < 0.85: slots["genre"] = book["genre"]
        for s in ["author", "rating_min", "price_max", "year_min"]:
            if r.random() < 0.22: slots[s] = self.value_from_book(s, book)
        if not slots: slots["genre"] = book["genre"]
        if allow_nomatch and r.random() < 0.07: slots["rating_min"] = "4.9"     # cố tình không có kết quả
        parts = [self.phrase(s, v) for s, v in slots.items()]
        utt = f"{r.choice(OPEN_ASK)} sách {' '.join(parts)}" + r.choice(["", "", " nhé", " giúp mình", " với"])
        return utt, slots

    def refine(self, state, rows):
        r = self.r
        cands = [s for s in ["rating_min", "price_max", "year_min", "author"] if s not in state]
        if not cands: return None
        s = r.choice(cands)
        v = self.value_from_book(s, r.choice(rows))
        if s == "rating_min" and r.random() < 0.1: v = "4.9"
        return f"{r.choice(REFINE_PRE)} {self.phrase(s, v)}", {s: v}

    def detail(self, n):
        r = self.r
        ask = r.choice(list(ASK_PHRASES))
        if n >= 2:
            idx = r.randint(1, min(n, 3))
            ref, turn = r.choice(ORDINALS[idx]), {"pick": str(idx), "ask": ask}
        else:
            ref, turn = r.choice(["cuốn này", "quyển này", "cuốn đó"]), {"ask": ask}
        utt = f"{ref.capitalize()} {r.choice(ASK_PHRASES[ask])}{r.choice([' vậy?', '?', ' bạn?'])}"
        return utt, turn

    def title_turn(self):
        r = self.r
        if r.random() < 0.1:
            t = r.choice(FAKE_TITLES)
        else:
            while True:
                t = r.choice(self.db.books)["title"]
                if len(self.db.query({"title": t.lower()})) == 1: break
        return r.choice(TITLE_Q).format(t=t), {"title": t.lower()}

    # ---------- chính sách hệ thống ----------
    def system(self, kind, bs, rows):
        r, n = self.r, len(rows)
        if kind == "greet":  return "request_genre", r.choice(T_GREET)
        if kind == "bye":    return "bye", r.choice(T_BYE)
        if kind == "detail": return "inform_detail", r.choice(T_DETAIL[bs["ask"]])
        if kind == "title":
            return ("inform_book", r.choice(T_BOOK)) if n else ("no_match_title", r.choice(T_NOTITLE))
        if n == 0: return "no_match", r.choice(T_NOMATCH)
        if n == 1: return "recommend", r.choice(T_REC1)
        if n == 2: return "recommend", r.choice(T_REC2)
        return "recommend", r.choice(T_REC3)

    # ---------- sinh 1 hội thoại ----------
    def generate_dialogue(self):
        r, db = self.r, self.db
        d, state, last_n = [], {}, 0

        def step(user, kind, turn_slots=None, reset=False):
            nonlocal state, last_n
            turn_slots = turn_slots or {}
            if reset or kind == "title": state = {}
            if kind == "search": state.pop("title", None)
            bs = dict(state); bs.update(turn_slots)
            rows = db.query(bs)
            act, tmpl = self.system(kind, bs, rows)
            if kind in ("search", "title"):
                state, last_n = dict(bs), len(rows)
            d.append(dict(user=user, kind=kind, bs=bs, n=len(rows), act=act, resp=tmpl,
                          resp_lex=lexicalize(tmpl, rows), row_ids=[x["id"] for x in rows]))

        b = r.choice(db.books)
        if r.random() < 0.2:
            step(r.choice(GREETS), "greet")
            utt, slots = self.opening(b); step(utt, "search", slots)
        else:
            utt, slots = self.opening(b)
            if r.random() < 0.25: utt = r.choice(["Chào bạn, ", "Xin chào, "]) + utt[0].lower() + utt[1:]
            step(utt, "search", slots)

        for _ in range(r.randint(1, 3)):
            if last_n == 0:                                   # không có kết quả -> người dùng tìm lại
                utt, slots = self.opening(r.choice(db.books), allow_nomatch=False)
                step(utt, "search", slots, reset=True); continue
            in_title = "title" in state
            action = r.choice(["detail"] * 4 + (["refine"] * 3 if not in_title else []) + ["new", "title"])
            if action == "refine":
                res = self.refine(state, db.query(state))
                if res is None: action = "detail"
                else: step(res[0], "search", res[1])
            if action == "detail":
                utt, ts = self.detail(last_n); step(utt, "detail", ts)
            elif action == "new":
                utt, slots = self.opening(r.choice(db.books), allow_nomatch=False)
                step(utt, "search", slots, reset=True)
            elif action == "title":
                utt, ts = self.title_turn(); step(utt, "title", ts)
        if r.random() < 0.85:
            step(r.choice(BYES), "bye")
        return d


gen = DialogueGenerator(db, seed=SEED)
dialogues = [gen.generate_dialogue() for _ in range(N_DIALOGUES)]
random.Random(SEED).shuffle(dialogues)
n_tr, n_va = int(0.8 * len(dialogues)), int(0.1 * len(dialogues))
train_d, val_d, test_d = dialogues[:n_tr], dialogues[n_tr:n_tr + n_va], dialogues[n_tr + n_va:]
print("Số hội thoại train/val/test:", len(train_d), len(val_d), len(test_d))
print("Số lượt:", sum(map(len, train_d)), sum(map(len, val_d)), sum(map(len, test_d)))
print("Phân bố act:", collections.Counter(t["act"] for d in dialogues for t in d).most_common())


def show_dialogue(d):
    for t in d:
        print("👤", t["user"])
        print("   bs =", t["bs"], "| n =", t["n"], "| act =", t["act"])
        print("🤖", t["resp_lex"], "\n")

show_dialogue(train_d[0]); print("-" * 80); show_dialogue(train_d[1])

Số hội thoại train/val/test: 1600 200 200
Số lượt: 6552 814 812
Phân bố act: [('recommend', 3758), ('bye', 1713), ('inform_detail', 1696), ('request_genre', 407), ('inform_book', 400), ('no_match', 169), ('no_match_title', 35)]
👤 Bạn có thể gợi ý sách tình cảm do Jeffrey Archer viết giúp mình
   bs = {'genre': 'tình cảm', 'author': 'Jeffrey Archer'} | n = 1 | act = recommend
🤖 Mình tìm thấy 1 cuốn phù hợp: Hai Số Phận của Jeffrey Archer, đánh giá 4.7/5 sao. Bạn có muốn xem chi tiết không? 

👤 Mình đang cần tìm sách về chủ đề lịch sử của tác giả Yuval Noah Harari ra mắt từ năm 2010 nhé
   bs = {'genre': 'lịch sử', 'author': 'Yuval Noah Harari', 'year_min': '2010'} | n = 1 | act = recommend
🤖 Mình tìm thấy 1 cuốn phù hợp: Sapiens Lược Sử Loài Người của Yuval Noah Harari, đánh giá 4.0/5 sao. Bạn có muốn xem chi tiết không? 

👤 Ok, thêm điều kiện đánh giá từ 3.5 sao trở lên
   bs = {'genre': 'lịch sử', 'author': 'Yuval Noah Harari', 'year_min': '2010', 'rating_min': '3.5'} | n = 1 | act = 

## 3. Chuyển dữ liệu sang dạng seq2seq
Một mô hình – hai tác vụ, phân biệt bằng prefix:

**Tác vụ 1 – Theo dõi trạng thái (belief state):**
```
input : dst: người dùng: ... hệ thống: ... người dùng: <câu hiện tại>
target: genre: trinh thám ; rating_min: 4.5
```
**Tác vụ 2 – Sinh hành động + câu trả lời (sau khi đã truy vấn DB):**
```
input : resp: <lịch sử> trạng thái: genre: trinh thám ; rating_min: 4.5 csdl: 3 kết quả
target: act: recommend resp: Mình tìm thấy [count] cuốn phù hợp. Top gợi ý: [title_1] của [author_1] ...
```
Khi huấn luyện dùng belief state/DB **chuẩn (gold)**; khi suy luận dùng belief state **do mô hình sinh** rồi truy vấn DB (đây là điểm "end-to-end": lỗi của bước 1 ảnh hưởng bước 2).

Lịch sử chỉ giữ tối đa 3 lượt gần nhất để giảm độ dài đầu vào.

In [6]:
def bs_to_str(bs):
    items = [f"{k}: {bs[k]}" for k in SLOT_ORDER if k in bs]
    return " ; ".join(items) if items else "trống"


def parse_bs(s):
    s = s.strip()
    if s in ("", "trống"): return {}
    bs = {}
    for part in s.split(";"):
        if ":" in part:
            k, v = part.split(":", 1)
            k, v = k.strip(), v.strip()
            if k in SLOT_ORDER and v: bs[k] = v
    return bs


def build_context(history, user, max_turns=3):
    parts = [f"người dùng: {u} hệ thống: {s}" for u, s in history[-max_turns:]]
    parts.append(f"người dùng: {user}")
    return " ".join(parts)


def make_resp_input(ctx, bs_str, n):
    return f"resp: {ctx} trạng thái: {bs_str} csdl: {n} kết quả"


def parse_output(text):
    m = re.match(r"\s*act:\s*(\S+)\s*resp:\s*(.*)", text, re.S)
    return (m.group(1), m.group(2).strip()) if m else ("unknown", text.strip())


def build_samples(dialogs):
    samples = []
    for d in dialogs:
        hist = []
        for t in d:
            ctx = build_context(hist, t["user"])
            bs_str = bs_to_str(t["bs"])
            samples.append({"input": "dst: " + ctx, "target": bs_str, "task": "dst"})
            samples.append({"input": make_resp_input(ctx, bs_str, t["n"]),
                            "target": f"act: {t['act']} resp: {t['resp']}", "task": "resp"})
            hist.append((t["user"], t["resp_lex"]))
    return samples


train_s, val_s, test_s = build_samples(train_d), build_samples(val_d), build_samples(test_d)
print(len(train_s), len(val_s), len(test_s))
for s in train_s[2:4]:
    print("INPUT :", s["input"]); print("TARGET:", s["target"], "\n")

13104 1628 1624
INPUT : dst: người dùng: Bạn có thể gợi ý sách tình cảm do Jeffrey Archer viết giúp mình hệ thống: Mình tìm thấy 1 cuốn phù hợp: Hai Số Phận của Jeffrey Archer, đánh giá 4.7/5 sao. Bạn có muốn xem chi tiết không? người dùng: Mình đang cần tìm sách về chủ đề lịch sử của tác giả Yuval Noah Harari ra mắt từ năm 2010 nhé
TARGET: genre: lịch sử ; author: Yuval Noah Harari ; year_min: 2010 

INPUT : resp: người dùng: Bạn có thể gợi ý sách tình cảm do Jeffrey Archer viết giúp mình hệ thống: Mình tìm thấy 1 cuốn phù hợp: Hai Số Phận của Jeffrey Archer, đánh giá 4.7/5 sao. Bạn có muốn xem chi tiết không? người dùng: Mình đang cần tìm sách về chủ đề lịch sử của tác giả Yuval Noah Harari ra mắt từ năm 2010 nhé trạng thái: genre: lịch sử ; author: Yuval Noah Harari ; year_min: 2010 csdl: 1 kết quả
TARGET: act: recommend resp: Mình tìm thấy 1 cuốn phù hợp: [title_1] của [author_1], đánh giá [rating_1]/5 sao. Bạn có muốn xem chi tiết không? 



## 4. Fine-tune mô hình seq2seq tiền huấn luyện
**Thiết lập trên Kaggle:** bật *Settings → Internet: On* (cần để tải mô hình từ Hugging Face) và *Accelerator: GPU T4 x2 hoặc P100*. Notebook chỉ dùng 1 GPU (`cuda:0`) nên không bị lỗi đa GPU. Kết quả lưu ở `/kaggle/working`.

Để tránh lỗi do khác phiên bản thư viện, phần huấn luyện dùng **vòng lặp PyTorch thuần** (không dùng `Trainer` / `datasets`). Cell nạp mô hình **tự thử lần lượt** ViT5 → mT5-small → BARTpho và dừng ở mô hình đầu tiên nạp được. Có cell **smoke test** chạy thử 1 bước trước khi train thật, nếu lỗi sẽ lỗi ngay ở đó.

In [8]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

print("torch", torch.__version__, "| transformers", transformers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(KHÔNG có GPU - hãy bật GPU trong Settings)")
torch.manual_seed(SEED)
# bf16 chỉ dùng cho GPU đời mới (Ampere trở lên). T4/P100 dùng fp32 cho ổn định với T5.
use_bf16 = device == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8


def load_model_and_tokenizer(names):
    last = None
    for name in dict.fromkeys(names):              # giữ thứ tự, bỏ trùng
        try:
            tok_ = AutoTokenizer.from_pretrained(name)
            mdl_ = AutoModelForSeq2SeqLM.from_pretrained(name)
            return name, tok_, mdl_
        except Exception as e:
            print(f"[!] Không nạp được '{name}': {type(e).__name__}: {str(e)[:300]}\n")
            last = e
    raise RuntimeError("Không nạp được mô hình nào. Kiểm tra: (1) Settings > Internet đã bật chưa; "
                       "(2) tên mô hình; (3) phiên bản transformers.") from last


MODEL_NAME, tok, model = load_model_and_tokenizer([MODEL_NAME, "google/mt5-small", "vinai/bartpho-syllable"])
model.to(device)
LR = 3e-4 if "t5" in MODEL_NAME.lower() else 5e-5
print(f"==> Đang dùng: {MODEL_NAME} | {sum(p.numel() for p in model.parameters())/1e6:.0f}M tham số | lr={LR}")

torch 2.11.0+cu130 | transformers 5.20.2
Device: cuda Tesla T4


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


==> Đang dùng: google/mt5-small | 300M tham số | lr=0.0003


In [9]:
# --- kiểm tra tokenizer mã hóa/giải mã đúng các ký tự đặc biệt ta dùng ([ ] ; : /) ---
sample_targets = [s["target"] for s in train_s[:400]]
bad = []
for t in sample_targets:
    back = tok.decode(tok(t)["input_ids"], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    if norm(back) != norm(t): bad.append((t, back))
print(f"Round-trip đúng: {100*(1-len(bad)/len(sample_targets)):.1f}%")
for t, b in bad[:3]: print("GỐC :", t, "\nGIẢI :", b, "\n")
# Nếu tỉ lệ thấp (< 95%): đổi placeholder [title_1] -> @title_1 trong các template rồi sinh lại dữ liệu.

Round-trip đúng: 100.0%


In [10]:
def encode(samples):
    return [(tok(s["input"], max_length=MAX_SRC, truncation=True)["input_ids"],
             tok(s["target"], max_length=MAX_TGT, truncation=True)["input_ids"]) for s in samples]

pad_id = tok.pad_token_id

def collate(batch):
    L, T = max(len(x) for x, _ in batch), max(len(y) for _, y in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    lab = torch.full((len(batch), T), -100, dtype=torch.long)       # -100: bỏ qua khi tính loss
    for i, (x, y) in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1; lab[i, :len(y)] = torch.tensor(y)
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

train_enc, val_enc = encode(train_s), encode(val_s)
print("Độ dài đầu vào (token) trung bình/tối đa:",
      round(np.mean([len(x) for x, _ in train_enc])), max(len(x) for x, _ in train_enc),
      "| đầu ra:", round(np.mean([len(y) for _, y in train_enc])), max(len(y) for _, y in train_enc))


@torch.no_grad()
def generate(prompts, max_new_tokens=MAX_TGT, num_beams=4, batch_size=32):
    model.eval(); outs = []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC)
        ids = model.generate(input_ids=enc["input_ids"].to(model.device),
                             attention_mask=enc["attention_mask"].to(model.device),
                             max_new_tokens=max_new_tokens, num_beams=num_beams)
        outs += tok.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return outs


# ---- SMOKE TEST: chạy thử 1 bước forward/backward + generate để phát hiện lỗi sớm ----
_b = {k: v.to(device) for k, v in collate(train_enc[:4]).items()}
model.train(); _loss = model(**_b).loss; _loss.backward(); model.zero_grad()
print("Loss ban đầu:", round(_loss.item(), 3), "(bình thường vài đơn vị; NaN là có vấn đề)")
print("Generate thử:", generate([train_s[0]["input"]], num_beams=1, max_new_tokens=20))
print("Smoke test OK ✔")

Độ dài đầu vào (token) trung bình/tối đa: 181 384 | đầu ra: 35 96
Loss ban đầu: 24.32 (bình thường vài đơn vị; NaN là có vấn đề)
Generate thử: ['<extra_id_0>.) <extra_id_37> Jeffrey Archer.']
Smoke test OK ✔


In [11]:
import time

def train_model(model, train_enc, val_enc, collate_fn, epochs, lr, batch_size, keep_best=True, bf16=False, log_every=100):
    tl = DataLoader(train_enc, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    vl = DataLoader(val_enc, batch_size=32, collate_fn=collate_fn) if val_enc else None
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    total = len(tl) * epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.05 * total), total)
    dev = next(model.parameters()).device
    best, best_state, t0 = float("inf"), None, time.time()
    for ep in range(1, epochs + 1): 
        model.train(); run = 0.0
        for step, b in enumerate(tl, 1):
            b = {k: v.to(dev) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=bf16):
                loss = model(**b).loss
            if not math.isfinite(loss.item()):
                raise RuntimeError("Loss = NaN/inf. Thử giảm LR, tắt bf16, hoặc đổi MODEL_NAME.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
            run += loss.item()
            if step % log_every == 0:
                print(f"  epoch {ep} | step {step}/{len(tl)} | loss {run/step:.4f} | {time.time()-t0:.0f}s")
        msg = f"==> Epoch {ep}/{epochs} | train loss {run/len(tl):.4f}"
        if vl is not None:
            model.eval(); tot = n = 0
            with torch.no_grad():
                for b in vl:
                    b = {k: v.to(dev) for k, v in b.items()}
                    tot += model(**b).loss.item() * b["input_ids"].size(0); n += b["input_ids"].size(0)
            val = tot / n; msg += f" | val loss {val:.4f}"
            if keep_best and val < best:
                best = val; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(msg)
    if best_state is not None:
        model.load_state_dict(best_state); print("Đã nạp lại checkpoint có val loss thấp nhất:", round(best, 4))
    return model


train_model(model, train_enc, val_enc, collate, epochs=EPOCHS, lr=LR, batch_size=BATCH, bf16=use_bf16)
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR + "/final"); tok.save_pretrained(OUT_DIR + "/final")
print("Đã lưu mô hình tại", OUT_DIR + "/final")

  epoch 1 | step 100/819 | loss 17.7064 | 86s
  epoch 1 | step 200/819 | loss 10.8390 | 179s
  epoch 1 | step 300/819 | loss 7.4748 | 275s
  epoch 1 | step 400/819 | loss 5.6943 | 369s
  epoch 1 | step 500/819 | loss 4.6012 | 465s
  epoch 1 | step 600/819 | loss 3.8601 | 561s
  epoch 1 | step 700/819 | loss 3.3263 | 656s
  epoch 1 | step 800/819 | loss 2.9233 | 752s
==> Epoch 1/5 | train loss 2.8579 | val loss 0.0406
  epoch 2 | step 100/819 | loss 0.0851 | 900s
  epoch 2 | step 200/819 | loss 0.0805 | 995s
  epoch 2 | step 300/819 | loss 0.0737 | 1091s
  epoch 2 | step 400/819 | loss 0.0702 | 1186s
  epoch 2 | step 500/819 | loss 0.0667 | 1281s
  epoch 2 | step 600/819 | loss 0.0633 | 1376s
  epoch 2 | step 700/819 | loss 0.0603 | 1471s
  epoch 2 | step 800/819 | loss 0.0575 | 1566s
==> Epoch 2/5 | train loss 0.0570 | val loss 0.0198
  epoch 3 | step 100/819 | loss 0.0346 | 1714s
  epoch 3 | step 200/819 | loss 0.0334 | 1808s
  epoch 3 | step 300/819 | loss 0.0334 | 1903s
  epoch 3 | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình tại /kaggle/working/tod-books/final


## 5. Suy luận & chatbot hội thoại
`BookChatbot.respond()` thực hiện đúng quy trình end-to-end:
1. Sinh **belief state** từ lịch sử + câu người dùng.
2. **Truy vấn DB** bằng belief state vừa sinh.
3. Sinh **act + câu trả lời delexicalized** (điều kiện theo belief state và số kết quả).
4. **Lexicalize**: điền thực thể thật từ DB.

In [12]:
class BookChatbot:
    def __init__(self, gen_fn, db, verbose=False):
        self.gen_fn, self.db, self.verbose = gen_fn, db, verbose
        self.history = []

    def reset(self):
        self.history = []

    def respond(self, user_utt):
        ctx = build_context(self.history, user_utt)
        bs_str = self.gen_fn(["dst: " + ctx])[0]                       # (1) belief state
        bs = parse_bs(bs_str)
        rows = self.db.query(bs)                                      # (2) truy vấn DB
        out = self.gen_fn([make_resp_input(ctx, bs_to_str(bs), len(rows))])[0]   # (3) act + response
        act, resp = parse_output(out)
        reply = lexicalize(resp, rows)                                # (4) điền thực thể
        self.history.append((user_utt, reply))
        if self.verbose:
            print(f"   [bs={bs} | n={len(rows)} | act={act}]")
        return reply


bot = BookChatbot(generate, db, verbose=True)
demo = ["Xin chào",
        "Gợi ý giúp mình sách trinh thám",
        "Lọc thêm giúp mình đánh giá từ 4.0 sao trở lên",
        "Cuốn thứ hai giá bao nhiêu vậy?",
        "Bạn có cuốn Đắc Nhân Tâm không?",
        "Cảm ơn nhé, tạm biệt"]
for u in demo:
    print("👤", u); print("🤖", bot.respond(u), "\n")

👤 Xin chào
   [bs={} | n=0 | act=request_genre]
🤖 Xin chào! Mình là trợ lý tìm sách. Bạn đang quan tâm thể loại nào ạ? 

👤 Gợi ý giúp mình sách trinh thám
   [bs={'genre': 'trinh thám'} | n=6 | act=recommend]
🤖 Có 6 cuốn đáp ứng yêu cầu của bạn, nổi bật là Bạch Dạ Hành (Higashino Keigo), Án Mạng Trên Chuyến Tàu Tốc Hành Phương Đông (Agatha Christie) và Nghi Can X (Higashino Keigo). Bạn cần mình lọc thêm điều kiện nào không? 

👤 Lọc thêm giúp mình đánh giá từ 4.0 sao trở lên
   [bs={'genre': 'trinh thám', 'rating_min': '4.0'} | n=5 | act=recommend]
🤖 Có 5 cuốn đáp ứng yêu cầu của bạn, nổi bật là Bạch Dạ Hành (Higashino Keigo), Án Mạng Trên Chuyến Tàu Tốc Hành Phương Đông (Agatha Christie) và Nghi Can X (Higashino Keigo). Bạn cần mình lọc thêm điều kiện nào không? 

👤 Cuốn thứ hai giá bao nhiêu vậy?
   [bs={'genre': 'trinh thám', 'rating_min': '4.0', 'pick': '2', 'ask': 'price'} | n=1 | act=inform_detail]
🤖 Cuốn Án Mạng Trên Chuyến Tàu Tốc Hành Phương Đông có giá 165 nghìn đồng. 

👤 Bạn 

## ns

In [13]:
def chat_cli():
    """Chat tương tác trong notebook. Gõ 'thoát' để dừng, 'reset' để bắt đầu lại."""
    bot = BookChatbot(generate, db, verbose=True)
    while True:
        u = input("👤 Bạn: ").strip()
        if u.lower() in ("thoát", "quit", "exit"): break
        if u.lower() == "reset": bot.reset(); print("(đã reset)"); continue
        print("🤖 Bot:", bot.respond(u), "\n")

chat_cli()   # bỏ comment để chat thử

👤 Bạn:  chào chatbot


   [bs={} | n=0 | act=request_genre]
🤖 Bot: Chào bạn! Bạn muốn mình gợi ý sách thể loại nào (văn học, trinh thám, kinh tế, tâm lý, ...)? 



👤 Bạn:  Có cuốn Hai Vạn Dặm Dưới Đáy Biển không?


   [bs={'title': 'hai vạn dặm dưới đáy biển'} | n=1 | act=inform_book]
🤖 Bot: Mình tìm thấy Hai Vạn Dặm Dưới Đáy Biển của Jules Verne, thuộc thể loại khoa học viễn tưởng, xuất bản năm 1870. Sách được đánh giá 4.7/5 sao, giá 85 nghìn đồng. 



👤 Bạn:  Bạn Có sách về thể loại tình cảm không?


   [bs={'genre': 'tình cảm'} | n=3 | act=recommend]
🤖 Bot: Có 3 cuốn đáp ứng yêu cầu của bạn, nổi bật là Cô Gái Đến Từ Hôm Qua (Nguyễn Nhật Ánh), Rừng Na Uy (Haruki Murakami) và Hai Số Phận (Jeffrey Archer). Bạn cần mình lọc thêm điều kiện nào không? 



👤 Bạn:  Có cuốn Cô Gái Đến Từ Hôm Qua (Nguyễn Nhật Ánh) không shop?


   [bs={'title': 'cô gái đến từ hôm qua'} | n=1 | act=inform_book]
🤖 Bot: Có ạ! Cô Gái Đến Từ Hôm Qua của Nguyễn Nhật Ánh (tình cảm, 1989), đánh giá 4.8/5, giá 170 nghìn đồng. 



👤 Bạn:  Giá tiền của cuốn Hai Số Phận. 


   [bs={'title': 'hai số phận'} | n=1 | act=inform_book]
🤖 Bot: Có ạ! Hai Số Phận của Jeffrey Archer (tình cảm, 1979), đánh giá 4.7/5, giá 155 nghìn đồng. 



👤 Bạn:  thoát


In [14]:
# (Tùy chọn) Giao diện web đơn giản bằng Gradio:  
!pip install -q gradio
import gradio as gr
_bot = BookChatbot(generate, db) 
def _fn(msg, hist): return _bot.respond(msg) 
gr.ChatInterface(_fn, title="Trợ lý tìm & gợi ý sách").launch(share=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://239055fcaa0f190f86.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Đánh giá
Đánh giá theo từng lượt trên tập test (lịch sử lấy từ hội thoại chuẩn), **toàn bộ qua pipeline end-to-end** (belief state do mô hình sinh → DB → câu trả lời):

| Chỉ số | Ý nghĩa |
|---|---|
| **JGA** (Joint Goal Accuracy) | % lượt mà belief state sinh ra **khớp hoàn toàn** với nhãn |
| **Slot-F1** | F1 trên các cặp (slot, value) |
| **Act-Acc** | % lượt chọn đúng dialogue act |
| **Inform** | % lượt truy vấn DB bằng belief state sinh ra trả về **đúng tập sách** như nhãn |
| **Success** | Inform đúng **và** (nếu người dùng hỏi thông tin cụ thể) câu trả lời có chứa đúng placeholder thông tin đó |
| **BLEU** | giữa câu trả lời delexicalized sinh ra và câu chuẩn (chỉ mang tính tham khảo vì có nhiều cách diễn đạt) |

In [15]:
def evaluate_system(dialogs, gen_fn, db):
    turns = []
    for d in dialogs:
        hist = []
        for t in d: 
            turns.append((build_context(hist, t["user"]), t))
            hist.append((t["user"], t["resp_lex"]))

    pred_bs = [parse_bs(s) for s in gen_fn(["dst: " + c for c, _ in turns])]
    pred_rows = [db.query(b) for b in pred_bs]
    outs = gen_fn([make_resp_input(c, bs_to_str(b), len(r)) for (c, _), b, r in zip(turns, pred_bs, pred_rows)])
    outs = [parse_output(o) for o in outs]

    jga, f1s, act_ok, inform, success = [], [], [], [], []
    for (ctx, t), pb, pr, (pact, presp) in zip(turns, pred_bs, pred_rows, outs):
        gold = set(t["bs"].items()); pred = set(pb.items())
        jga.append(float(pb == t["bs"]))
        tp = len(gold & pred)
        p, r_ = tp / max(len(pred), 1), tp / max(len(gold), 1)
        f1s.append(1.0 if not gold and not pred else (2 * p * r_ / (p + r_) if p + r_ else 0.0))
        act_ok.append(float(pact == t["act"]))
        inf = [x["id"] for x in pr] == t["row_ids"]
        inform.append(float(inf))
        need = f"[{t['bs']['ask']}_1]" if "ask" in t["bs"] else None
        success.append(float(inf and (need is None or need in presp)))

    try:
        import sacrebleu
        bleu = sacrebleu.corpus_bleu([o[1] for o in outs], [[t["resp"] for _, t in turns]]).score
    except ImportError:
        bleu = float("nan")

    res = {"turns": len(turns), "JGA": np.mean(jga), "Slot-F1": np.mean(f1s), "Act-Acc": np.mean(act_ok),
           "Inform": np.mean(inform), "Success": np.mean(success), "BLEU": bleu}
    details = [dict(user=t["user"], gold_bs=t["bs"], pred_bs=pb, gold_act=t["act"], pred_act=pa, pred_resp=pr_)
               for (_, t), pb, (pa, pr_) in zip(turns, pred_bs, outs)]
    return res, details


def print_results(res):
    for k, v in res.items():
        print(f"{k:8s}: {v:.4f}" if isinstance(v, float) else f"{k:8s}: {v}")

In [16]:
# ---- Sanity check: "mô hình oracle" phải đạt gần điểm tuyệt đối -> chứng tỏ code đánh giá đúng ----
_lookup = {s["input"]: s["target"] for s in test_s}
oracle_gen = lambda prompts: [_lookup[p] for p in prompts]
res_oracle, _ = evaluate_system(test_d, oracle_gen, db)
print_results(res_oracle)

turns   : 812
JGA     : 1.0000
Slot-F1 : 1.0000
Act-Acc : 1.0000
Inform  : 1.0000
Success : 1.0000
BLEU    : 98.6429


In [17]:
res, details = evaluate_system(test_d, generate, db)
print_results(res)
os.makedirs(OUT_DIR, exist_ok=True)
json.dump({k: float(v) for k, v in res.items()}, open(OUT_DIR + "/test_results.json", "w"), ensure_ascii=False, indent=2)

turns   : 812
JGA     : 0.9581
Slot-F1 : 0.9813
Act-Acc : 0.9951
Inform  : 0.9729
Success : 0.9729
BLEU    : 71.0393


In [ ]:
# ---- Phân tích lỗi ----
errors = [d for d in details if d["pred_bs"] != d["gold_bs"]]
print(f"{len(errors)}/{len(details)} lượt sai belief state\n")
for e in errors[:8]:
    print("USER  :", e["user"]); print("GOLD  :", e["gold_bs"]); print("PRED  :", e["pred_bs"], "\n")

# Độ chính xác theo từng slot
slot_stat = collections.defaultdict(lambda: [0, 0])
for e in details:
    for k, v in e["gold_bs"].items():
        slot_stat[k][1] += 1
        slot_stat[k][0] += int(e["pred_bs"].get(k) == v)
for k, (c, n) in sorted(slot_stat.items()):
    print(f"{k:11s} {c}/{n} = {c/n:.3f}")

## 7. (Tùy chọn) Fine-tune LLM decoder-only bằng LoRA
Cùng định dạng dữ liệu `dst:` / `resp:` nhưng dùng mô hình **decoder-only** (Qwen, Llama, ...) với **LoRA** để tiết kiệm bộ nhớ. Đặt `RUN_LLM = True` và cài `peft`. Cell này chỉ khác ở chỗ tạo prompt (`input` + `### Trả lời:` + `target`) và mask loss trên phần prompt; sau khi train, truyền hàm `generate_llm` vào `BookChatbot` / `evaluate_system` y như mô hình seq2seq → so sánh công bằng giữa T5/BART và LLM.

> Với Llama cần xin quyền truy cập trên Hugging Face và `huggingface-cli login`. Qwen2.5 không cần.

In [18]:
RUN_LLM = False          # đặt True để chạy (phần này tùy chọn, chưa được kiểm thử trên Kaggle)
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # hoặc "meta-llama/Llama-3.2-1B-Instruct"

if RUN_LLM:
    # !pip install -q peft 
    from transformers import AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model

    llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
    if llm_tok.pad_token is None: llm_tok.pad_token = llm_tok.eos_token
    llm_tok.padding_side = "left"                    # cần cho generate
    llm_dtype = torch.bfloat16 if use_bf16 else torch.float32     # T4/P100: fp32 + LoRA cho ổn định
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=llm_dtype).to(device)
    llm = get_peft_model(llm, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    llm.print_trainable_parameters()

    SEP, LLM_MAX = "\n### Trả lời:\n", 512
    def causal_enc(s):
        p = llm_tok(s["input"] + SEP, add_special_tokens=False)["input_ids"]
        t = llm_tok(s["target"] + llm_tok.eos_token, add_special_tokens=False)["input_ids"]
        return {"ids": (p + t)[:LLM_MAX], "lab": ([-100] * len(p) + t)[:LLM_MAX]}    # chỉ tính loss trên phần target

    def causal_collate(batch):
        L = max(len(x["ids"]) for x in batch)
        ids = torch.full((len(batch), L), llm_tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        for i, x in enumerate(batch):
            n = len(x["ids"]); ids[i, :n] = torch.tensor(x["ids"]); att[i, :n] = 1; lab[i, :n] = torch.tensor(x["lab"])
        return {"input_ids": ids, "attention_mask": att, "labels": lab}

    train_llm = [causal_enc(s) for s in train_s]
    train_model(llm, train_llm, None, causal_collate, epochs=2, lr=2e-4, batch_size=8, keep_best=False, bf16=use_bf16)

    @torch.no_grad()
    def generate_llm(prompts, max_new_tokens=MAX_TGT, batch_size=16):
        llm.eval(); outs = []
        for i in range(0, len(prompts), batch_size):
            b = llm_tok([p + SEP for p in prompts[i:i + batch_size]], return_tensors="pt", padding=True,
                        add_special_tokens=False).to(llm.device)
            ids = llm.generate(**b, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=llm_tok.pad_token_id)
            outs += [llm_tok.decode(x[b["input_ids"].shape[1]:], skip_special_tokens=True).strip() for x in ids]
        return outs

    res_llm, _ = evaluate_system(test_d, generate_llm, db)
    print_results(res_llm)

## 8. Đánh giá người dùng cho môn HCI (System Usability Scale)
Vì đề tài thuộc môn **HCI**, ngoài chỉ số tự động nên làm thêm một **user study nhỏ** (5–15 người):
1. Giao 3–5 nhiệm vụ (vd. *"tìm sách trinh thám đánh giá ≥ 4.0, giá dưới 150k và cho biết tác giả cuốn đầu tiên"*).
2. Ghi lại **tỉ lệ hoàn thành**, **số lượt hội thoại**, **thời gian**, số lần hệ thống hiểu sai.
3. Cho điền bảng câu hỏi **SUS** (10 câu, thang 1–5) và tính điểm bên dưới (≥ 68 được xem là trên mức trung bình).

In [19]:
SUS_ITEMS = [
    "Tôi nghĩ mình sẽ muốn sử dụng hệ thống này thường xuyên.",
    "Tôi thấy hệ thống phức tạp một cách không cần thiết.",
    "Tôi thấy hệ thống dễ sử dụng.",
    "Tôi nghĩ mình cần sự hỗ trợ của người có kỹ thuật để dùng được hệ thống này.",
    "Tôi thấy các chức năng trong hệ thống được tích hợp tốt.",
    "Tôi thấy hệ thống có quá nhiều điểm không nhất quán.",
    "Tôi nghĩ hầu hết mọi người sẽ học cách dùng hệ thống này rất nhanh.",
    "Tôi thấy hệ thống rất rườm rà, khó dùng.",
    "Tôi cảm thấy tự tin khi sử dụng hệ thống.",
    "Tôi cần học nhiều thứ trước khi có thể sử dụng hệ thống này.",
]

def sus_score(answers):
    """answers: 10 số nguyên 1-5 theo thứ tự câu hỏi. Trả về điểm SUS 0-100."""
    assert len(answers) == 10
    odd  = sum(a - 1 for a in answers[0::2])      # câu 1,3,5,7,9 (tích cực)
    even = sum(5 - a for a in answers[1::2])      # câu 2,4,6,8,10 (tiêu cực)
    return (odd + even) * 2.5

# Ví dụ: 3 người tham gia
participants = [[4, 2, 4, 1, 4, 2, 5, 2, 4, 1],
                [5, 1, 5, 2, 4, 1, 4, 1, 5, 2],
                [3, 3, 4, 2, 3, 2, 4, 2, 3, 3]]
scores = [sus_score(p) for p in participants]
print("Điểm SUS từng người:", scores, "| Trung bình:", round(sum(scores) / len(scores), 1))

Điểm SUS từng người: [82.5, 90.0, 62.5] | Trung bình: 78.3


## 9. Hướng mở rộng & gợi ý cho báo cáo
* **Dữ liệu thật**: thay KB mô phỏng bằng dữ liệu sách thật (ví dụ *Goodbooks-10k*, hoặc crawl từ nhà sách có cho phép), thêm thể loại/mô tả; thu thập hội thoại thật (Wizard-of-Oz) hoặc dùng LLM sinh hội thoại rồi **kiểm duyệt thủ công** để tăng độ đa dạng so với template.
* **Đa dạng ngôn ngữ**: thêm câu không dấu, viết tắt, lỗi chính tả, câu hỏi gộp nhiều ý, tham chiếu "cuốn vừa rồi".
* **Mô hình**: so sánh ViT5 vs BARTpho vs mT5 vs LLM+LoRA (mục 7); thử *belief-state delta* thay vì tích lũy đầy đủ; thêm *retrieval-augmented* (đưa top-k sách vào prompt thay vì chỉ số lượng).
* **Chính sách hội thoại**: hỏi lại khi thiếu thông tin (clarification), gợi ý thay thế khi không có kết quả, xử lý yêu cầu ngoài miền (out-of-domain).
* **HCI**: A/B test giao diện (chat thuần vs chat + nút gợi ý nhanh), đo SUS, thời gian hoàn thành nhiệm vụ, phỏng vấn sau thử nghiệm, phân tích các điểm vỡ hội thoại (breakdown).
* **Cảnh báo trung thực khoa học**: điểm đánh giá trên dữ liệu mô phỏng thường rất cao (vì cùng phân phối template); hãy tạo thêm **tập test "khó"** (câu diễn đạt khác, nhiều ý) và/hoặc test với người thật để báo cáo kết quả thực tế hơn.

In [20]:
# ===== DEMO GIAO DIỆN: Trang chủ BookStore + khung chat gắn với mô hình TOD =====
import html
import ipywidgets as W
from IPython.display import display

GEN = generate          # đổi thành generate_llm nếu bạn dùng LLM + LoRA ở mục 7
bot = BookChatbot(GEN, db)

# ---------------------------------------------------------------
# 1) TRANG CHỦ BOOKSTORE (bám theo ảnh bạn gửi)
# ---------------------------------------------------------------
# (tên, giá, màu bìa, link ảnh bìa hoặc None). Muốn dùng ảnh thật: điền URL/data-URI vào phần tử thứ 4.
PRODUCTS = [
    ("Harry Potter và hòn đá phù thủy", "116.000 VNĐ", "linear-gradient(160deg,#e74c3c,#8e44ad)", None),
    ("Harry Potter và hoàng tử lai",    "120.000 VNĐ", "linear-gradient(160deg,#3498db,#1b4f8a)", None),
    ("Harry Potter và tên tù nhân",     "135.000 VNĐ", "linear-gradient(160deg,#2c3e70,#1a2a4f)", None),
    ("Harry Potter và hội phượng",      "150.000 VNĐ", "linear-gradient(160deg,#9b7fc4,#5b3f8c)", None),
    ("Harry Potter và phòng chứa",      "120.000 VNĐ", "linear-gradient(160deg,#8e7cc3,#4a3a7a)", None),
]

def product_card(name, price, grad, img):
    bg = f"url('{img}') center/cover" if img else grad
    return f"""
    <div style="width:122px;background:#f4f5fb;border-radius:12px;box-shadow:0 2px 6px #0002;overflow:hidden;flex:none;">
      <div style="height:92px;background:{bg};color:#fff;font-weight:700;font-size:11px;padding:6px;text-align:center;
                  display:flex;align-items:center;justify-content:center;">{'' if img else name}</div>
      <div style="padding:8px 8px 4px;font-size:11px;line-height:1.3;height:48px;overflow:hidden;">{name}</div>
      <div style="padding:0 8px;font-size:13px;font-weight:700;">{price}</div>
      <div style="padding:2px 8px;color:#2a5db0;font-size:11px;letter-spacing:1px;">★★★★☆</div>
      <div style="margin:6px 8px 10px;padding:6px 0;border-radius:16px;background:linear-gradient(#42a5f5,#1e88e5);
                  color:#fff;text-align:center;font-size:11px;font-weight:600;">Add to cart</div>
    </div>"""

cards = "".join(product_card(*p) for p in PRODUCTS)

PAGE_HTML = f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;background:#eef0f8;color:#111;border:1px solid #bbb;border-radius:10px;overflow:hidden;width:720px;">
  <!-- thanh thông tin -->
  <div style="display:flex;align-items:flex-start;gap:14px;padding:10px 16px 0;font-size:12px;">
    <div>Hệ thống cửa hàng<br>0998365474</div><div style="font-size:18px">📍</div>
    <span>Về địa chỉ</span><span>Tin Tức</span><span>Sự kiện</span>
    <span style="margin-left:auto;margin-right:40px;">Thông tin liên hệ</span>
  </div>
  <!-- logo + tìm kiếm + icon -->
  <div style="display:flex;align-items:center;gap:18px;padding:4px 16px 10px;border-bottom:2px solid #111;">
    <div style="text-align:center;font-weight:700;font-size:11px;"><div style="font-size:24px;line-height:1">📖</div>BookStore</div>
    <div style="flex:1;max-width:300px;padding:9px 16px;border-radius:22px;background:#fff;box-shadow:0 2px 8px #0002;color:#888;font-size:13px;">🔍 Search</div>
    <div style="margin-left:auto;display:flex;gap:20px;font-size:11px;font-weight:600;text-align:center;">
      <span style="font-size:18px">🛒<br><span style="font-size:11px">Giỏ hàng</span></span>
      <span style="font-size:18px">♡<br><span style="font-size:11px">Yêu thích</span></span>
      <span style="font-size:18px">👤<br><span style="font-size:11px">Tài khoản</span></span>
    </div>
  </div>
  <!-- menu -->
  <div style="display:flex;align-items:center;gap:22px;padding:12px 16px;font-weight:600;font-size:12px;">
    <span style="border:1px solid #ccc;padding:9px 18px;background:#fff;">☰ &nbsp; Danh mục</span>
    <span>Sách mới</span><span>Sách xu hướng</span><span>Pre order sản phẩm</span>
    <span>Ấn phẩm đặc biệt</span><span>Sắp phát hành</span><span>Bảng tin</span>
  </div>
  <!-- banner -->
  <div style="margin:0 16px;height:130px;border-radius:6px;background:linear-gradient(110deg,#f6e8d6,#e6c9a0 60%,#d9b98f);
              display:flex;align-items:center;padding:0 34px;font-family:Georgia,serif;font-size:30px;color:#6b4a14;line-height:1.15;">
    THẾ GIỚI QUA<br>TỪNG TRANG SÁCH
  </div>
  <!-- sản phẩm -->
  <div style="font-size:22px;font-weight:700;margin:16px 16px 10px 34px;">Sách sắp phát hành</div>
  <div style="display:flex;gap:12px;padding:0 16px 18px;">{cards}</div>
</div>"""
page = W.HTML(PAGE_HTML, layout=W.Layout(width="720px"))

# ---------------------------------------------------------------
# 2) KHUNG CHAT gắn với pipeline TOD: belief state -> truy vấn DB -> act + câu trả lời
# ---------------------------------------------------------------
def respond_dbg(user_utt):
    """Giống BookChatbot.respond nhưng trả thêm thông tin debug để hiển thị."""
    ctx = build_context(bot.history, user_utt)
    bs = parse_bs(GEN(["dst: " + ctx])[0])                               # (1) belief state
    rows = db.query(bs)                                                  # (2) truy vấn DB
    out = GEN([make_resp_input(ctx, bs_to_str(bs), len(rows))])[0]       # (3) act + response
    act, resp = parse_output(out)
    reply = lexicalize(resp, rows)                                       # (4) điền thực thể
    bot.history.append((user_utt, reply))
    return reply, f"bs: {bs_to_str(bs)} · {len(rows)} kết quả · act: {act}"

WELCOME = "Xin chào! Mình là trợ lý tìm sách. Bạn muốn tìm thể loại, tác giả hay tên sách nào ạ?"
log = [("bot", WELCOME, None)]

view = W.HTML(layout=W.Layout(width="100%"))
box = W.Text(placeholder="Nhập tin nhắn...", continuous_update=False, layout=W.Layout(flex="1 1 auto"))
send_btn = W.Button(description="Gửi ➤", button_style="danger", layout=W.Layout(width="80px"))
show_dbg = W.Checkbox(value=True, description="Hiện belief state (debug)", indent=False)
reset_btn = W.Button(description="↺ Hội thoại mới", layout=W.Layout(width="130px"))

def render(pending=False):
    rows = ""
    for who, txt, info in log:
        t = html.escape(txt)
        if who == "me":
            rows += (f'<div style="display:flex;justify-content:flex-end;margin:5px 0"><div style="max-width:86%;'
                     f'background:linear-gradient(135deg,#d6247f,#ec4899);color:#fff;padding:8px 12px;'
                     f'border-radius:14px 14px 2px 14px;font-size:13px;white-space:pre-line">{t}</div></div>')
        else:
            dbg = (f'<div style="font-size:10px;color:#999;margin-top:4px;font-family:monospace">{html.escape(info)}</div>'
                   if (info and show_dbg.value) else "")
            rows += (f'<div style="display:flex;margin:5px 0"><div style="max-width:86%;background:#fff;border:1px solid #eee;'
                     f'color:#111;padding:8px 12px;border-radius:14px 14px 14px 2px;font-size:13px;white-space:pre-line">{t}{dbg}</div></div>')
    if pending:
        rows += '<div style="color:#999;font-size:12px;margin:4px 6px">Đang nhập…</div>'
    # column-reverse -> khung tự cuộn tới tin mới nhất
    view.value = ('<div style="display:flex;flex-direction:column-reverse;height:400px;overflow-y:auto;'
                  f'background:#faf7fb;padding:8px;font-family:Segoe UI,Arial,sans-serif"><div>{rows}</div></div>')

def send(text):
    text = (text or "").strip()
    if not text:
        return
    log.append(("me", text, None))
    box.disabled = send_btn.disabled = True
    render(pending=True)
    try:
        reply, info = respond_dbg(text)
    except Exception as e:
        reply, info = f"⚠️ Lỗi khi sinh câu trả lời: {e}", None
    log.append(("bot", reply, info))
    box.disabled = send_btn.disabled = False
    render()

def on_send(_=None):
    v = box.value; box.value = ""
    send(v)

def on_reset(_=None):
    bot.reset(); log.clear(); log.append(("bot", WELCOME, None)); render()

send_btn.on_click(on_send); box.on_submit(on_send); reset_btn.on_click(on_reset)
show_dbg.observe(lambda _: render(), names="value")

# các nút gợi ý nhanh
QUICK = ["Gợi ý giúp mình sách trinh thám",
         "Lọc thêm giúp mình đánh giá từ 4.0 sao trở lên",
         "Cuốn thứ hai giá bao nhiêu vậy?",
         "Bạn có cuốn Đắc Nhân Tâm không?",
         "Cảm ơn nhé, tạm biệt"]
chips = []
for q in QUICK:
    b = W.Button(description=q if len(q) < 28 else q[:26] + "…", tooltip=q, layout=W.Layout(margin="2px"))
    b.on_click(lambda _, q=q: send(q))
    chips.append(b)

header = W.HTML(
    '<div style="background:linear-gradient(135deg,#d6247f,#ec4899);color:#fff;padding:10px 14px;'
    'border-radius:12px 12px 0 0;font-family:Segoe UI,Arial,sans-serif;">'
    '<b style="font-size:15px">💬 Chat với Mely</b><br><small>Luôn sẵn sàng hỗ trợ</small></div>',
    layout=W.Layout(width="100%"))
footer = W.HTML('<div style="text-align:center;font-size:11px;color:#888">Powered by ViT5/T5 · end-to-end TOD</div>')

chat = W.VBox(
    [header, view,
     W.HBox(chips, layout=W.Layout(flex_flow="row wrap", width="100%")),
     W.HBox([box, send_btn]),
     W.HBox([show_dbg, reset_btn], layout=W.Layout(justify_content="space-between", padding="2px 6px")),
     footer],
    layout=W.Layout(width="340px", border="1px solid #ddd", border_radius="12px", margin="0 0 0 16px"))

render()
display(W.HBox([page, chat], layout=W.Layout(align_items="flex-start"))) 

/tmp/ipykernel_49/1729205579.py:135: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  send_btn.on_click(on_send); box.on_submit(on_send); reset_btn.on_click(on_reset)


In [21]:
# ===== CHÈN ẢNH THẬT VÀO TRANG BOOKSTORE =====
import base64, io, glob
from PIL import Image

IMG_DIR = "/kaggle/input/datasets/rtnhiuthanhthao/hinhanhchosach"     # đổi theo tên dataset của bạn (hoặc "/kaggle/working/images")

def find(name):
    fs = glob.glob(f"{IMG_DIR}/**/{name}.*", recursive=True)
    return fs[0] if fs else None

def img_uri(path, max_size, q=82):
    """Thu nhỏ ảnh rồi mã hóa base64 để nhúng thẳng vào HTML (không cần Internet, không phụ thuộc đường dẫn)."""
    im = Image.open(path).convert("RGB")
    im.thumbnail(max_size)
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=q)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

# 1) Bìa sách: cover1..cover5
new_products, found = [], 0
for i, (name, price, grad, _) in enumerate(PRODUCTS, 1):
    f = find(f"cover{i}")
    found += bool(f)
    new_products.append((name, price, grad, img_uri(f, (300, 400)) if f else None))   # không có ảnh -> giữ ô màu
new_cards = "".join(product_card(*p) for p in new_products)
html_new = PAGE_HTML.replace(cards, new_cards)

# 2) Banner
fb = find("banner")
if fb:
    html_new = html_new.replace("background:linear-gradient(110deg,#f6e8d6,#e6c9a0 60%,#d9b98f);",
                                f"background:url('{img_uri(fb, (1400, 300))}') center/cover;")
    html_new = html_new.replace("THẾ GIỚI QUA<br>TỪNG TRANG SÁCH", "")   # ảnh banner đã có sẵn chữ

page.value = html_new
print(f"Đã nạp {found}/{len(PRODUCTS)} ảnh bìa | banner: {'có' if fb else 'không tìm thấy'}") 

Đã nạp 0/5 ảnh bìa | banner: không tìm thấy


In [22]:
# ===== DEMO GIAO DIỆN: Trang chủ BookStore + khung chat gắn với mô hình TOD =====
import html
import ipywidgets as W
from IPython.display import display

GEN = generate          # đổi thành generate_llm nếu bạn dùng LLM + LoRA ở mục 7
bot = BookChatbot(GEN, db)

# ---------------------------------------------------------------
# 1) TRANG CHỦ BOOKSTORE (bám theo ảnh bạn gửi)
# ---------------------------------------------------------------
# (tên, giá, màu bìa, link ảnh bìa hoặc None). Muốn dùng ảnh thật: điền URL/data-URI vào phần tử thứ 4.
PRODUCTS = [
    ("Harry Potter và hòn đá phù thủy", "116.000 VNĐ", "linear-gradient(160deg,#e74c3c,#8e44ad)", None),
    ("Harry Potter và hoàng tử lai",    "120.000 VNĐ", "linear-gradient(160deg,#3498db,#1b4f8a)", None),
    ("Harry Potter và tên tù nhân",     "135.000 VNĐ", "linear-gradient(160deg,#2c3e70,#1a2a4f)", None),
    ("Harry Potter và hội phượng",      "150.000 VNĐ", "linear-gradient(160deg,#9b7fc4,#5b3f8c)", None),
    ("Harry Potter và phòng chứa",      "120.000 VNĐ", "linear-gradient(160deg,#8e7cc3,#4a3a7a)", None),
]

def product_card(name, price, grad, img):
    bg = f"url('{img}') center/cover" if img else grad
    return f"""
    <div style="width:122px;background:#f4f5fb;border-radius:12px;box-shadow:0 2px 6px #0002;overflow:hidden;flex:none;">
      <div style="height:92px;background:{bg};color:#fff;font-weight:700;font-size:11px;padding:6px;text-align:center;
                  display:flex;align-items:center;justify-content:center;">{'' if img else name}</div>
      <div style="padding:8px 8px 4px;font-size:11px;line-height:1.3;height:48px;overflow:hidden;">{name}</div>
      <div style="padding:0 8px;font-size:13px;font-weight:700;">{price}</div>
      <div style="padding:2px 8px;color:#2a5db0;font-size:11px;letter-spacing:1px;">★★★★☆</div>
      <div style="margin:6px 8px 10px;padding:6px 0;border-radius:16px;background:linear-gradient(#42a5f5,#1e88e5);
                  color:#fff;text-align:center;font-size:11px;font-weight:600;">Add to cart</div>
    </div>"""

cards = "".join(product_card(*p) for p in PRODUCTS)

PAGE_HTML = f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;background:#eef0f8;color:#111;border:1px solid #bbb;border-radius:10px;overflow:hidden;width:720px;">
  <!-- thanh thông tin -->
  <div style="display:flex;align-items:flex-start;gap:14px;padding:10px 16px 0;font-size:12px;">
    <div>Hệ thống cửa hàng<br>0998365474</div><div style="font-size:18px">📍</div>
    <span>Về địa chỉ</span><span>Tin Tức</span><span>Sự kiện</span>
    <span style="margin-left:auto;margin-right:40px;">Thông tin liên hệ</span>
  </div>
  <!-- logo + tìm kiếm + icon -->
  <div style="display:flex;align-items:center;gap:18px;padding:4px 16px 10px;border-bottom:2px solid #111;">
    <div style="text-align:center;font-weight:700;font-size:11px;"><div style="font-size:24px;line-height:1">📖</div>BookStore</div>
    <div style="flex:1;max-width:300px;padding:9px 16px;border-radius:22px;background:#fff;box-shadow:0 2px 8px #0002;color:#888;font-size:13px;">🔍 Search</div>
    <div style="margin-left:auto;display:flex;gap:20px;font-size:11px;font-weight:600;text-align:center;">
      <span style="font-size:18px">🛒<br><span style="font-size:11px">Giỏ hàng</span></span>
      <span style="font-size:18px">♡<br><span style="font-size:11px">Yêu thích</span></span>
      <span style="font-size:18px">👤<br><span style="font-size:11px">Tài khoản</span></span>
    </div>
  </div>
  <!-- menu -->
  <div style="display:flex;align-items:center;gap:22px;padding:12px 16px;font-weight:600;font-size:12px;">
    <span style="border:1px solid #ccc;padding:9px 18px;background:#fff;">☰ &nbsp; Danh mục</span>
    <span>Sách mới</span><span>Sách xu hướng</span><span>Pre order sản phẩm</span>
    <span>Ấn phẩm đặc biệt</span><span>Sắp phát hành</span><span>Bảng tin</span>
  </div>
  <!-- banner -->
  <div style="margin:0 16px;height:130px;border-radius:6px;background:linear-gradient(110deg,#f6e8d6,#e6c9a0 60%,#d9b98f);
              display:flex;align-items:center;padding:0 34px;font-family:Georgia,serif;font-size:30px;color:#6b4a14;line-height:1.15;">
    THẾ GIỚI QUA<br>TỪNG TRANG SÁCH
  </div>
  <!-- sản phẩm -->
  <div style="font-size:22px;font-weight:700;margin:16px 16px 10px 34px;">Sách sắp phát hành</div>
  <div style="display:flex;gap:12px;padding:0 16px 18px;">{cards}</div>
</div>"""
page = W.HTML(PAGE_HTML, layout=W.Layout(width="720px"))

# ---------------------------------------------------------------
# 2) KHUNG CHAT gắn với pipeline TOD: belief state -> truy vấn DB -> act + câu trả lời
# ---------------------------------------------------------------
def respond_dbg(user_utt):
    """Giống BookChatbot.respond nhưng trả thêm thông tin debug để hiển thị."""
    ctx = build_context(bot.history, user_utt)
    bs = parse_bs(GEN(["dst: " + ctx])[0])                               # (1) belief state
    rows = db.query(bs)                                                  # (2) truy vấn DB
    out = GEN([make_resp_input(ctx, bs_to_str(bs), len(rows))])[0]       # (3) act + response
    act, resp = parse_output(out)
    reply = lexicalize(resp, rows)                                       # (4) điền thực thể
    bot.history.append((user_utt, reply))
    return reply, f"bs: {bs_to_str(bs)} · {len(rows)} kết quả · act: {act}"

WELCOME = "Xin chào! Mình là trợ lý tìm sách. Bạn muốn tìm thể loại, tác giả hay tên sách nào ạ?"
log = [("bot", WELCOME, None)]

view = W.HTML(layout=W.Layout(width="100%"))
box = W.Text(placeholder="Nhập tin nhắn...", continuous_update=False, layout=W.Layout(flex="1 1 auto"))
send_btn = W.Button(description="Gửi ➤", button_style="danger", layout=W.Layout(width="80px"))
show_dbg = W.Checkbox(value=True, description="Hiện belief state (debug)", indent=False)
reset_btn = W.Button(description="↺ Hội thoại mới", layout=W.Layout(width="130px"))

def render(pending=False):
    rows = ""
    for who, txt, info in log:
        t = html.escape(txt)
        if who == "me":
            rows += (f'<div style="display:flex;justify-content:flex-end;margin:5px 0"><div style="max-width:86%;'
                     f'background:linear-gradient(135deg,#d6247f,#ec4899);color:#fff;padding:8px 12px;'
                     f'border-radius:14px 14px 2px 14px;font-size:13px;white-space:pre-line">{t}</div></div>')
        else:
            dbg = (f'<div style="font-size:10px;color:#999;margin-top:4px;font-family:monospace">{html.escape(info)}</div>'
                   if (info and show_dbg.value) else "")
            rows += (f'<div style="display:flex;margin:5px 0"><div style="max-width:86%;background:#fff;border:1px solid #eee;'
                     f'color:#111;padding:8px 12px;border-radius:14px 14px 14px 2px;font-size:13px;white-space:pre-line">{t}{dbg}</div></div>')
    if pending:
        rows += '<div style="color:#999;font-size:12px;margin:4px 6px">Đang nhập…</div>'
    # column-reverse -> khung tự cuộn tới tin mới nhất
    view.value = ('<div style="display:flex;flex-direction:column-reverse;height:400px;overflow-y:auto;'
                  f'background:#faf7fb;padding:8px;font-family:Segoe UI,Arial,sans-serif"><div>{rows}</div></div>')

def send(text):
    text = (text or "").strip()
    if not text:
        return
    log.append(("me", text, None))
    box.disabled = send_btn.disabled = True
    render(pending=True)
    try:
        reply, info = respond_dbg(text)
    except Exception as e:
        reply, info = f"⚠️ Lỗi khi sinh câu trả lời: {e}", None
    log.append(("bot", reply, info))
    box.disabled = send_btn.disabled = False
    render()

def on_send(_=None):
    v = box.value; box.value = ""
    send(v)

def on_reset(_=None):
    bot.reset(); log.clear(); log.append(("bot", WELCOME, None)); render()

send_btn.on_click(on_send); box.on_submit(on_send); reset_btn.on_click(on_reset)
show_dbg.observe(lambda _: render(), names="value")

# các nút gợi ý nhanh
QUICK = ["Gợi ý giúp mình sách trinh thám",
         "Lọc thêm giúp mình đánh giá từ 4.0 sao trở lên",
         "Cuốn thứ hai giá bao nhiêu vậy?",
         "Bạn có cuốn Đắc Nhân Tâm không?",
         "Cảm ơn nhé, tạm biệt"]
chips = []
for q in QUICK:
    b = W.Button(description=q if len(q) < 28 else q[:26] + "…", tooltip=q, layout=W.Layout(margin="2px"))
    b.on_click(lambda _, q=q: send(q))
    chips.append(b)

header = W.HTML(
    '<div style="background:linear-gradient(135deg,#d6247f,#ec4899);color:#fff;padding:10px 14px;'
    'border-radius:12px 12px 0 0;font-family:Segoe UI,Arial,sans-serif;">'
    '<b style="font-size:15px">💬 Chat với Mely</b><br><small>Luôn sẵn sàng hỗ trợ</small></div>',
    layout=W.Layout(width="100%"))
footer = W.HTML('<div style="text-align:center;font-size:11px;color:#888">Powered by ViT5/T5 · end-to-end TOD</div>')

chat = W.VBox(
    [header, view,
     W.HBox(chips, layout=W.Layout(flex_flow="row wrap", width="100%")),
     W.HBox([box, send_btn]),
     W.HBox([show_dbg, reset_btn], layout=W.Layout(justify_content="space-between", padding="2px 6px")),
     footer],
    layout=W.Layout(width="340px", border="1px solid #ddd", border_radius="12px", margin="0 0 0 16px"))

render()
display(W.HBox([page, chat], layout=W.Layout(align_items="flex-start")))

/tmp/ipykernel_49/3302339952.py:135: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  send_btn.on_click(on_send); box.on_submit(on_send); reset_btn.on_click(on_reset)


In [25]:
# ===== DEMO GIAO DIỆN: Trang chủ BookStore (có ảnh bìa) + khung chat gắn với mô hình TOD =====
import html, base64, io, glob, os, re, unicodedata, warnings
import ipywidgets as W
from IPython.display import display
from PIL import Image

warnings.filterwarnings("ignore", category=DeprecationWarning)   # ẩn cảnh báo on_submit (vô hại)

GEN = generate          # đổi thành generate_llm nếu bạn dùng LLM + LoRA ở mục 7
bot = BookChatbot(GEN, db)

# ---------------------------------------------------------------
# 0) TÌM ẢNH TRONG /kaggle/input (tự ghép theo từ khóa trong tên file)
# ---------------------------------------------------------------
IMG_ROOT = "/kaggle/input/datasets/rtnhiuthanhthao/hinhanhsach"
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
ALL_IMGS = [p for p in glob.glob(f"{IMG_ROOT}/**/*", recursive=True) if p.lower().endswith(IMG_EXT)]

def fold(s):
    """Bỏ dấu, chữ thường, thay _ - bằng khoảng trắng -> dễ so khớp tên file."""
    s = unicodedata.normalize("NFD", s.lower().replace("đ", "d")) 
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[\W_]+", " ", s).strip() 

def find_img(keywords):
    for p in ALL_IMGS:
        name = fold(os.path.basename(p))
        if any(k in name for k in keywords):
            return p
    return None

def img_uri(path, max_size, q=85):
    """Thu nhỏ rồi mã hóa base64 để nhúng thẳng vào HTML (không phụ thuộc đường dẫn/Internet)."""
    im = Image.open(path).convert("RGB")
    im.thumbnail(max_size)
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=q)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

# ---------------------------------------------------------------
# 1) TRANG CHỦ BOOKSTORE
# ---------------------------------------------------------------
# (tên, giá, màu dự phòng khi chưa có ảnh, danh sách từ khóa để tìm ảnh trong tên file)
PRODUCTS = [
    ("Harry Potter và hòn đá phù thủy", "116.000 VNĐ", "linear-gradient(160deg,#e74c3c,#8e44ad)", ["ph thy", "phu thuy", "hon da"]),
    ("Harry Potter và hoàng tử lai",    "120.000 VNĐ", "linear-gradient(160deg,#3498db,#1b4f8a)", ["t lai", "tu lai"]),
    ("Harry Potter và tên tù nhân",     "135.000 VNĐ", "linear-gradient(160deg,#2c3e70,#1a2a4f)", ["tu nhan", "t nhn", "azkaban"]),
    ("Harry Potter và hội phượng",      "150.000 VNĐ", "linear-gradient(160deg,#9b7fc4,#5b3f8c)", ["phng hong", "phuong hoang"]),
    ("Harry Potter và phòng chứa",      "120.000 VNĐ", "linear-gradient(160deg,#8e7cc3,#4a3a7a)", ["phng cha", "phong chua"]),
]

def product_card(name, price, grad, img):
    if img:      # có ảnh bìa -> hiển thị ảnh, khung cao hơn
        cover = f'<div style="height:150px;background:#fff url(\'{img}\') center/cover no-repeat;"></div>'
    else:        # chưa có ảnh -> ô màu + tên sách
        cover = (f'<div style="height:92px;background:{grad};color:#fff;font-weight:700;font-size:11px;padding:6px;'
                 f'text-align:center;display:flex;align-items:center;justify-content:center;">{name}</div>')
    return f"""
    <div style="width:122px;background:#f4f5fb;border-radius:12px;box-shadow:0 2px 6px #0002;overflow:hidden;flex:none;">
      {cover}
      <div style="padding:8px 8px 4px;font-size:11px;line-height:1.3;height:48px;overflow:hidden;">{name}</div>
      <div style="padding:0 8px;font-size:13px;font-weight:700;">{price}</div>
      <div style="padding:2px 8px;color:#2a5db0;font-size:11px;letter-spacing:1px;">★★★★☆</div>
      <div style="margin:6px 8px 10px;padding:6px 0;border-radius:16px;background:linear-gradient(#42a5f5,#1e88e5);
                  color:#fff;text-align:center;font-size:11px;font-weight:600;">Add to cart</div>
    </div>"""

cards_html, n_found = "", 0
for name, price, grad, kws in PRODUCTS:
    f = find_img(kws)
    n_found += bool(f)
    cards_html += product_card(name, price, grad, img_uri(f, (300, 400)) if f else None)

# Banner: nếu có file tên chứa "banner" thì dùng ảnh, không thì dùng nền gradient + chữ
fb = find_img(["banner"])
if fb:
    BANNER_BG, BANNER_TEXT = f"url('{img_uri(fb, (1400, 300))}') center/cover", ""
else:
    BANNER_BG, BANNER_TEXT = "linear-gradient(110deg,#f6e8d6,#e6c9a0 60%,#d9b98f)", "THẾ GIỚI QUA<br>TỪNG TRANG SÁCH"

PAGE_HTML = f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;background:#eef0f8;color:#111;border:1px solid #bbb;border-radius:10px;overflow:hidden;width:720px;">
  <div style="display:flex;align-items:flex-start;gap:14px;padding:10px 16px 0;font-size:12px;">
    <div>Hệ thống cửa hàng<br>0998365474</div><div style="font-size:18px">📍</div>
    <span>Về địa chỉ</span><span>Tin Tức</span><span>Sự kiện</span>
    <span style="margin-left:auto;margin-right:40px;">Thông tin liên hệ</span>
  </div>
  <div style="display:flex;align-items:center;gap:18px;padding:4px 16px 10px;border-bottom:2px solid #111;">
    <div style="text-align:center;font-weight:700;font-size:11px;"><div style="font-size:24px;line-height:1">📖</div>BookStore</div>
    <div style="flex:1;max-width:300px;padding:9px 16px;border-radius:22px;background:#fff;box-shadow:0 2px 8px #0002;color:#888;font-size:13px;">🔍 Search</div>
    <div style="margin-left:auto;display:flex;gap:20px;font-size:11px;font-weight:600;text-align:center;">
      <span style="font-size:18px">🛒<br><span style="font-size:11px">Giỏ hàng</span></span>
      <span style="font-size:18px">♡<br><span style="font-size:11px">Yêu thích</span></span>
      <span style="font-size:18px">👤<br><span style="font-size:11px">Tài khoản</span></span>
    </div>
  </div>
  <div style="display:flex;align-items:center;gap:22px;padding:12px 16px;font-weight:600;font-size:12px;">
    <span style="border:1px solid #ccc;padding:9px 18px;background:#fff;">☰ &nbsp; Danh mục</span>
    <span>Sách mới</span><span>Sách xu hướng</span><span>Pre order sản phẩm</span>
    <span>Ấn phẩm đặc biệt</span><span>Sắp phát hành</span><span>Bảng tin</span>
  </div>
  <div style="margin:0 16px;height:130px;border-radius:6px;background:{BANNER_BG};
              display:flex;align-items:center;padding:0 34px;font-family:Georgia,serif;font-size:30px;color:#6b4a14;line-height:1.15;">
    {BANNER_TEXT}
  </div>
  <div style="font-size:22px;font-weight:700;margin:16px 16px 10px 34px;">Sách sắp phát hành</div>
  <div style="display:flex;gap:12px;padding:0 16px 18px;">{cards_html}</div>
</div>"""
page = W.HTML(PAGE_HTML, layout=W.Layout(width="720px"))
print(f"Ảnh bìa: {n_found}/{len(PRODUCTS)} | banner: {'có' if fb else 'không có (dùng nền mặc định)'}")

# ---------------------------------------------------------------
# 2) KHUNG CHAT gắn với pipeline TOD: belief state -> truy vấn DB -> act + câu trả lời
# ---------------------------------------------------------------
def respond_dbg(user_utt):
    ctx = build_context(bot.history, user_utt)
    bs = parse_bs(GEN(["dst: " + ctx])[0])                               # (1) belief state
    rows = db.query(bs)                                                  # (2) truy vấn DB
    out = GEN([make_resp_input(ctx, bs_to_str(bs), len(rows))])[0]       # (3) act + response
    act, resp = parse_output(out)
    reply = lexicalize(resp, rows)                                       # (4) điền thực thể
    bot.history.append((user_utt, reply))
    return reply, f"bs: {bs_to_str(bs)} · {len(rows)} kết quả · act: {act}"

WELCOME = "Xin chào! Mình là trợ lý tìm sách. Bạn muốn tìm thể loại, tác giả hay tên sách nào ạ?"
log = [("bot", WELCOME, None)]

view = W.HTML(layout=W.Layout(width="100%"))
box = W.Text(placeholder="Nhập tin nhắn...", continuous_update=False, layout=W.Layout(flex="1 1 auto"))
send_btn = W.Button(description="Gửi ➤", button_style="danger", layout=W.Layout(width="80px"))
show_dbg = W.Checkbox(value=True, description="Hiện belief state (debug)", indent=False)
reset_btn = W.Button(description="↺ Hội thoại mới", layout=W.Layout(width="130px"))

def render(pending=False):
    rows = ""
    for who, txt, info in log:
        t = html.escape(txt)
        if who == "me":
            rows += (f'<div style="display:flex;justify-content:flex-end;margin:5px 0"><div style="max-width:86%;'
                     f'background:linear-gradient(135deg,#d6247f,#ec4899);color:#fff;padding:8px 12px;'
                     f'border-radius:14px 14px 2px 14px;font-size:13px;white-space:pre-line">{t}</div></div>')
        else:
            dbg = (f'<div style="font-size:10px;color:#999;margin-top:4px;font-family:monospace">{html.escape(info)}</div>'
                   if (info and show_dbg.value) else "")
            rows += (f'<div style="display:flex;margin:5px 0"><div style="max-width:86%;background:#fff;border:1px solid #eee;'
                     f'color:#111;padding:8px 12px;border-radius:14px 14px 14px 2px;font-size:13px;white-space:pre-line">{t}{dbg}</div></div>')
    if pending:
        rows += '<div style="color:#999;font-size:12px;margin:4px 6px">Đang nhập…</div>'
    view.value = ('<div style="display:flex;flex-direction:column-reverse;height:400px;overflow-y:auto;'
                  f'background:#faf7fb;padding:8px;font-family:Segoe UI,Arial,sans-serif"><div>{rows}</div></div>')

def send(text):
    text = (text or "").strip()
    if not text:
        return
    log.append(("me", text, None))
    box.disabled = send_btn.disabled = True
    render(pending=True)
    try:
        reply, info = respond_dbg(text)
    except Exception as e:
        reply, info = f"⚠️ Lỗi khi sinh câu trả lời: {e}", None
    log.append(("bot", reply, info))
    box.disabled = send_btn.disabled = False
    render()

def on_send(_=None):
    v = box.value; box.value = ""
    send(v)

def on_reset(_=None):
    bot.reset(); log.clear(); log.append(("bot", WELCOME, None)); render()

send_btn.on_click(on_send); box.on_submit(on_send); reset_btn.on_click(on_reset)
show_dbg.observe(lambda _: render(), names="value")

QUICK = ["Gợi ý giúp mình sách trinh thám",
         "Lọc thêm giúp mình đánh giá từ 4.0 sao trở lên",
         "Cuốn thứ hai giá bao nhiêu vậy?",
         "Bạn có cuốn Đắc Nhân Tâm không?",
         "Cảm ơn nhé, tạm biệt"]
chips = []
for q in QUICK:
    b = W.Button(description=q if len(q) < 28 else q[:26] + "…", tooltip=q, layout=W.Layout(margin="2px"))
    b.on_click(lambda _, q=q: send(q))
    chips.append(b)

header = W.HTML(
    '<div style="background:linear-gradient(135deg,#d6247f,#ec4899);color:#fff;padding:10px 14px;'
    'border-radius:12px 12px 0 0;font-family:Segoe UI,Arial,sans-serif;">'
    '<b style="font-size:15px">💬 Chat với Mely</b><br><small>Luôn sẵn sàng hỗ trợ</small></div>',
    layout=W.Layout(width="100%"))
footer = W.HTML('<div style="text-align:center;font-size:11px;color:#888">Powered by ViT5/T5 · end-to-end TOD</div>')

chat = W.VBox(
    [header, view,
     W.HBox(chips, layout=W.Layout(flex_flow="row wrap", width="100%")),
     W.HBox([box, send_btn]),
     W.HBox([show_dbg, reset_btn], layout=W.Layout(justify_content="space-between", padding="2px 6px")),
     footer],
    layout=W.Layout(width="340px", border="1px solid #ddd", border_radius="12px", margin="0 0 0 16px"))

render()
display(W.HBox([page, chat], layout=W.Layout(align_items="flex-start")))

Ảnh bìa: 5/5 | banner: không có (dùng nền mặc định)
